In [3]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("practice").getOrCreate()

In [5]:
shipments_df = spark.read.csv(
    "shipments_75000.csv",
    header=True,
    inferSchema=True
)


**PART A | DataFrame Operations**

In [6]:
#1
shipments_df.printSchema()
print("Total number of records:",shipments_df.count())
shipments_df.show(15)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total number of records: 31366
+-----------+------------+-------------+----

In [7]:
#2
shipments_df.select(
    'shipment_id',
    'warehouse_id',
    'service_level',
    'shipment_status',
    'shipping_fee_inr'
).show()

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
| SHP0000001|         W08|      Express|      Delivered|          554.34|
| SHP0000002|         W08|     Standard|      Delivered|          414.56|
| SHP0000003|         W01|      Express|      Delivered|          724.64|
| SHP0000004|         W03|     Standard|      Delivered|          287.86|
| SHP0000005|         W03|      Economy|      Delivered|          251.68|
| SHP0000006|         W01|      Express|      Delivered|           477.9|
| SHP0000007|         W02|     Standard|     In Transit|          298.68|
| SHP0000008|         W03|      Economy|     In Transit|          284.52|
| SHP0000009|         W02|     Standard|      Delivered|          861.74|
| SHP0000010|         W02|      Economy|      Delivered|          309.76|
| SHP0000011|         W04|     Standar

In [8]:
#3
from pyspark.sql.functions import col
shipments_df.filter(
    (col('service_level')== 'Express')&
    (col('package_weight_kg') > 5)
).show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000003|  2026-01-31|   2026-02-01|            2026-02-03| 

In [9]:
#4
shipments_df.sort(
    col('shipping_fee_inr').desc()
).limit(20).show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0003419|  2026-01-28|   2026-01-31|            2026-02-05| 

**PART B | Aggregations and Joins**

In [10]:
#5
from pyspark.sql.functions import count
shipments_df.groupBy('shipment_status').agg(
    count('shipment_id').alias("Total Shipments")
).show()

+---------------+---------------+
|shipment_status|Total Shipments|
+---------------+---------------+
|       Returned|           2590|
|           Lost|            928|
|     In Transit|           4385|
|      Cancelled|           1521|
|      Delivered|          21942|
+---------------+---------------+



In [38]:
from pyspark.sql.functions import col, sum, when

clean_df = shipments_df.fillna(
    0, subset=['shipping_cost_inr']
)
service_summary = clean_df.groupBy('service_level').agg(
    sum('shipping_fee_inr').alias('total_fees'),
    sum('shipping_cost_inr').alias('total_costs'),
    (sum(col('shipping_fee_inr') - col('shipping_cost_inr'))
    ).alias('total_profit')
)
service_summary = service_summary.withColumn(
    'profit_margin_pct',
    when(
        col('total_fees') != 0,
        col('total_profit') / col('total_fees') * 100
    )
)

service_summary.show()

+-------------+------------------+------------------+------------------+------------------+
|service_level|        total_fees|       total_costs|      total_profit| profit_margin_pct|
+-------------+------------------+------------------+------------------+------------------+
|      Express| 4202214.960000024|3071191.9000000004| 1131023.059999995|26.914926313050593|
|     Standard|  5627854.08000004| 4221638.499999994|1406215.5800000043| 24.98671003211217|
|      Economy|1866043.4399999941|1410069.5000000028| 455973.9400000005|24.435333616885252|
+-------------+------------------+------------------+------------------+------------------+



In [12]:
warehouse_df = spark.read.csv(
    "warehouse_master.csv",
    header=True,
    inferSchema=True
)

In [13]:
#7
shipment_and_warehouse_df = shipments_df.join(
    warehouse_df,
    on = 'warehouse_id',
    how = 'inner'
)
shipment_and_warehouse_df.groupBy('region').agg(
    count('shipment_id').alias("Total Shipments")
).show()

+------+---------------+
|region|Total Shipments|
+------+---------------+
| South|          13939|
|  East|           2208|
|  West|          11193|
| North|           4026|
+------+---------------+



In [14]:
carrier_df = spark.read.csv(
    "carrier_master.csv",
    header=True,
    inferSchema=True
)
routes_df = spark.read.csv(
    'route_master.csv',
    header=True,
    inferSchema=True
)


In [15]:
#8
from pyspark.sql.functions import avg
ship_carrier_route_df = shipments_df.join(
    carrier_df,
    on = 'carrier_id',
    how = 'inner'
).join(
    routes_df,
    on = 'route_id',
    how = 'inner'
)

ship_carrier_route_df = ship_carrier_route_df.filter(
    (col('distance_km')!=0) &
    (col('shipping_cost_inr').isNotNull())
)
ship_carrier_route_df = ship_carrier_route_df.withColumn(
    'shipping_cost_per_km',
    col('shipping_cost_inr')/col('distance_km')
)
ship_carrier_route_df.groupBy('carrier_id','carrier_name').agg(
    avg(col('shipping_cost_per_km')).alias("Total Cost per KM")
).show()

+----------+--------------------+------------------+
|carrier_id|        carrier_name| Total Cost per KM|
+----------+--------------------+------------------+
|       C07|   BluePeak Delivery|0.6119066205953229|
|       C04|  AirSprint Couriers|0.6284058521974323|
|       C06|           RouteWise|0.5812105728973956|
|       C03|MetroRoute Logistics|0.6018024871348346|
|       C02|        ParcelBridge|0.6024508581546609|
|       C05|           CargoNest|0.5822784494814289|
|       C01|   SwiftLine Express|0.5928811549037508|
+----------+--------------------+------------------+



**PART C | Null Handling and Conditional Logic**

In [40]:
#9
from pyspark.sql.functions import col, sum, when

shipments_df.select(
    sum(when(col('carrier_id').isNull(), 1).otherwise(0))
        .alias('null_carrier_id'),
    sum(when(col('package_weight_kg').isNull(), 1).otherwise(0))
        .alias('null_package_weight'),
    sum(when(col('shipping_cost_inr').isNull(), 1).otherwise(0))
        .alias('null_shipping_cost'),
    sum(when(col('customer_rating').isNull(), 1).otherwise(0))
        .alias('null_customer_rating')
).show()

+---------------+-------------------+------------------+--------------------+
|null_carrier_id|null_package_weight|null_shipping_cost|null_customer_rating|
+---------------+-------------------+------------------+--------------------+
|            353|                469|               586|               10991|
+---------------+-------------------+------------------+--------------------+



In [54]:
#10
temp_df = ship_carrier_route_df.na.fill(
    0,
    subset=['handling_minutes']
)

temp_df2 = ship_carrier_route_df.na.drop(subset=['carrier_id'])



In [18]:
#11
from pyspark.sql.functions import when
ship_carrier_route_df = ship_carrier_route_df.withColumn(
    'weight_category',
    when(col('package_weight_kg') < 2, 'Light')
    .when((col('package_weight_kg') >= 2) & (col('package_weight_kg') <= 10), 'Medium')
    .when(col('package_weight_kg') > 10, 'Heavy')
    .otherwise('Unknown')
)
ship_carrier_route_df.show()

+--------+----------+-----------+------------+-------------+----------------------+--------------------+------------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+--------------------+------------+----------------+-----------------+-------------+------------------+-------------+-----------+--------+--------------------+---------------+
|route_id|carrier_id|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|        carrier_name|carrier_type|base_cost_per_km|reliability_score|origin_region|destination_region|   route_type|distance_km|toll_inr|shipping_cost_per_km|weight_category|
+--------+----------+-------

**PART D | Date and Time Functions**

In [41]:
from pyspark.sql.functions import col, to_date, datediff

dated_df = shipments_df.withColumn(
    'actual_delivery_date',
    to_date('actual_delivery_date')
).withColumn(
    'dispatch_date',
    to_date('dispatch_date')
)

delivered_df = dated_df.filter(
    col('shipment_status') == 'Delivered'
)

delivered_df.withColumn(
    'delivery_duration_days',
    datediff('actual_delivery_date', 'dispatch_date')
).show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+----------------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|delivery_duration_days|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+-------------------

In [20]:
#13
ship_carrier_route_df = ship_carrier_route_df.withColumn(
    'delivery_performance',
    when(col('actual_delivery_date') == col('promised_delivery_date'), 'On Time')
    .when(col('actual_delivery_date') > col('promised_delivery_date'), 'Late')
    .when(col('actual_delivery_date') < col('promised_delivery_date'), 'Early')
    .otherwise('Not yet delivered')
)
ship_carrier_route_df.show()

+--------+----------+-----------+------------+-------------+----------------------+--------------------+------------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+--------------------+------------+----------------+-----------------+-------------+------------------+-------------+-----------+--------+--------------------+---------------+--------------------+
|route_id|carrier_id|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|        carrier_name|carrier_type|base_cost_per_km|reliability_score|origin_region|destination_region|   route_type|distance_km|toll_inr|shipping_cost_per_km|weight_category|delivery

In [21]:
#14
from pyspark.sql.functions import month
months_df = ship_carrier_route_df.filter(
    col('shipment_status') == 'Delivered'
).withColumn(
    'month',
    month(col('booking_date'))
)
months_df.groupBy('month').agg(
    count(col('shipment_id')).alias("Total Shipments"),
    sum(col('shipping_fee_inr')).alias("Total Fees")
).show()

+-----+---------------+------------------+
|month|Total Shipments|        Total Fees|
+-----+---------------+------------------+
|    1|           2759|1040845.9199999996|
|    6|           2721|1001559.8799999995|
|    3|           2753| 1012123.600000002|
|    5|           2756|1028555.6199999986|
|    4|           2571| 966618.8600000001|
|    8|           2486| 919883.9599999995|
|    7|           2784|1035217.8799999972|
|    2|           2484| 925841.9400000012|
+-----+---------------+------------------+



**PART E | Window Functions**

In [42]:
from pyspark.sql.functions import col, sum, dense_rank
from pyspark.sql.window import Window

region_carrier_fees = shipments_df.join(
    warehouse_df,
    on='warehouse_id',
    how='inner'
).groupBy(
    'region', 'carrier_id'
).agg(
    sum('shipping_fee_inr').alias('total_fees')
)

window_spec = Window.partitionBy('region').orderBy(
    col('total_fees').desc()
)

region_carrier_fees.withColumn(
    'dense_rank',
    dense_rank().over(window_spec)
).orderBy(
    'region', 'dense_rank'
).show()

+------+----------+------------------+----------+
|region|carrier_id|        total_fees|dense_rank|
+------+----------+------------------+----------+
|  East|       C02|157422.92000000007|         1|
|  East|       C01|         156652.36|         2|
|  East|       C03|129579.09999999989|         3|
|  East|       C05|102622.10000000005|         4|
|  East|       C07| 91602.35999999997|         5|
|  East|       C04|          91390.42|         6|
|  East|       C06| 79740.79999999999|         7|
|  East|      NULL|           11235.3|         8|
| North|       C02|312340.89999999997|         1|
| North|       C01|         254863.76|         2|
| North|       C03|253285.31999999972|         3|
| North|       C05|195467.24000000002|         4|
| North|       C07|170111.73999999976|         5|
| North|       C04|164276.22000000012|         6|
| North|       C06|136818.95999999996|         7|
| North|      NULL|          16592.38|         8|
| South|       C02| 1054345.199999997|         1|


In [23]:
#16
from pyspark.sql.functions import row_number, col
window_spec = Window.partitionBy('warehouse_id').orderBy(
    col('booking_date').desc()
)
row_number_df = ship_carrier_route_df.withColumn(
    'row_number',
    row_number().over(window_spec)
)
row_number_df.filter(
    col('row_number') == 1
).show()


+--------+----------+-----------+------------+-------------+----------------------+--------------------+------------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+--------------------+------------+----------------+-----------------+-------------+------------------+-------------+-----------+--------+--------------------+---------------+--------------------+----------+
|route_id|carrier_id|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|        carrier_name|carrier_type|base_cost_per_km|reliability_score|origin_region|destination_region|   route_type|distance_km|toll_inr|shipping_cost_per_km|weight_catego

In [43]:
#17
from pyspark.sql.functions import sum, col
window_spec = Window.partitionBy('warehouse_id').orderBy(
    col('booking_date').asc(),
    col('shipment_id').asc()
)

running_total = ship_carrier_route_df.withColumn(
    'running_total',
    sum(col('shipping_fee_inr')).over(window_spec)
)
running_total.show()

+--------+----------+-----------+------------+-------------+----------------------+--------------------+------------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+--------------------+------------+----------------+-----------------+-------------+------------------+-------------+-----------+--------+--------------------+---------------+--------------------+------------------+
|route_id|carrier_id|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|        carrier_name|carrier_type|base_cost_per_km|reliability_score|origin_region|destination_region|   route_type|distance_km|toll_inr|shipping_cost_per_km|weigh

**PART F | Union and Complex JSON**

In [25]:
extra_df = spark.read.csv(
    'shipments_extra_12000.csv',
    header=True,
    inferSchema=True
)

In [26]:
#18
combined_df = shipments_df.unionByName(extra_df)
combined_df.count()

43366

In [49]:
#19
print("Count before deduplication:", combined_df.count())
combined_df.groupBy(col('shipment_id')).count().filter(col('count') > 1).show()
combined_df = combined_df.dropDuplicates(subset=['shipment_id'])
print("Count after deduplication:", combined_df.count())

Count before deduplication: 43056
+-----------+-----+
|shipment_id|count|
+-----------+-----+
+-----------+-----+

Count after deduplication: 43056


In [28]:
sensor_df = spark.read.json(
    'warehouse_sensor_config.json',
    multiLine=True
)

In [29]:
#20
from pyspark.sql.functions import explode
sensor_df.select(
    'warehouse_id',
    'environment.temperature_range_c.minimum',
    'environment.temperature_range_c.maximum',
    explode('sensors').alias('sensor')
    ).show()

+------------+-------+-------+--------------------+
|warehouse_id|minimum|maximum|              sensor|
+------------+-------+-------+--------------------+
|         W01|     18|     32|{false, W01-S1, T...|
|         W01|     18|     32|{true, W01-S2, Hu...|
|         W01|     18|     32|{true, W01-S3, Door}|
|         W02|     18|     32|{true, W02-S1, Te...|
|         W02|     18|     32|{true, W02-S2, Hu...|
|         W02|     18|     32|{true, W02-S3, Door}|
|         W03|     18|     32|{true, W03-S1, Te...|
|         W03|     18|     32|{true, W03-S2, Hu...|
|         W03|     18|     32|{true, W03-S3, Door}|
|         W04|     18|     32|{true, W04-S1, Te...|
|         W04|     18|     32|{true, W04-S2, Hu...|
|         W04|     18|     32|{false, W04-S3, D...|
|         W05|     18|     32|{true, W05-S1, Te...|
|         W05|     18|     32|{false, W05-S2, H...|
|         W05|     18|     32|{true, W05-S3, Door}|
|         W06|     16|     24|{false, W06-S1, T...|
|         W0

**PART G | Lazy Execution, Caching, and Persistence**

In [50]:
#21
filtered_df = shipments_df.filter(
    col('shipment_status') == 'Delivered'
)
join_df = filtered_df.join(
    warehouse_df,
    on = 'warehouse_id',
    how = 'inner'
)
add_column = join_df.withColumn(
    'delivery_duration',
    col('actual_delivery_date') - col('dispatch_date')
)
grouped_df = add_column.groupBy('region').agg(
    count(col('shipment_id')).alias("Total Shipments"),
    sum(col('shipping_fee_inr')).alias("Total Fees")
)
ordering_df = grouped_df.orderBy('region')

ordering_df.explain()

ordering_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [region#435 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(region#435 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=6268]
      +- HashAggregate(keys=[region#435], functions=[count(shipment_id#17), sum(shipping_fee_inr#28)])
         +- Exchange hashpartitioning(region#435, 200), ENSURE_REQUIREMENTS, [plan_id=6265]
            +- HashAggregate(keys=[region#435], functions=[partial_count(shipment_id#17), partial_sum(shipping_fee_inr#28)])
               +- Project [shipment_id#17, shipping_fee_inr#28, region#435]
                  +- BroadcastHashJoin [warehouse_id#22], [warehouse_id#432], Inner, BuildRight, false
                     :- Filter isnotnull(warehouse_id#22)
                     :  +- InMemoryTableScan [shipment_id#17, warehouse_id#22, shipping_fee_inr#28], [isnotnull(warehouse_id#22)]
                     :        +- InMemoryRelation [shipment_id#17, booking_date#18, dispatch_date#19, promis

In [31]:
#22
import time
delivered_df = shipments_df.filter(
    col('shipment_status') == 'Delivered'
)

delivered_df.cache()

start_time = time.time()
delivered_df.count()
end_time = time.time()
print("Count 1 time:",end_time - start_time)

start_time = time.time()
delivered_df.count()
end_time = time.time()
print("Count 2 time:",end_time - start_time)

#action
delivered_df.groupBy('service_level').agg(
    sum(col('shipping_fee_inr')).alias("Total Fees"),
    sum(col('shipping_cost_inr')).alias("Total Costs")).show()



Count 1 time: 1.6757187843322754
Count 2 time: 0.15207266807556152
+-------------+------------------+-----------------+
|service_level|        Total Fees|      Total Costs|
+-------------+------------------+-----------------+
|      Express|2938909.6600000057|2153111.399999997|
|     Standard|3923313.6799999867|2948761.500000009|
|      Economy|1299311.9399999997|981565.3000000017|
+-------------+------------------+-----------------+



In [51]:
#23
from pyspark import StorageLevel
from pyspark.sql.functions import col

express_shipment = shipments_df.filter(
    col('service_level') == 'Express'
)

express_shipment.persist(StorageLevel.MEMORY_AND_DISK)

express_shipment.count()
express_shipment.show()

print(express_shipment.storageLevel)
express_shipment.unpersist()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000001|  2026-05-24|   2026-05-24|            2026-05-26| 

DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

**PART H | Partitioning and Optimization**

In [33]:
#24
from pyspark.sql.functions import spark_partition_id

shipment_df8 = shipments_df.repartition(8)

shipment_df8.groupBy(spark_partition_id().alias('partition_id')).count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3920|
|           1| 3920|
|           2| 3921|
|           3| 3921|
|           4| 3921|
|           5| 3921|
|           6| 3921|
|           7| 3921|
+------------+-----+



In [34]:
#25
shipment_df12 = shipment_df8.repartition(12)

shipment_df3_coalesce = shipment_df12.coalesce(3)

shipment_df3_repartition = shipment_df12.repartition(3)

shipment_df3_coalesce.explain()
shipment_df3_repartition.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=3268]
      +- FileScan csv [shipment_id#17,booking_date#18,dispatch_date#19,promised_delivery_date#20,actual_delivery_date#21,warehouse_id#22,carrier_id#23,route_id#24,customer_segment#25,service_level#26,package_weight_kg#27,shipping_fee_inr#28,shipping_cost_inr#29,shipment_status#30,delivery_attempts#31,damage_flag#32,customer_rating#33,handling_minutes#34,payment_mode#35,priority_flag#36] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/shipments_75000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<shipment_id:string,booking_date:date,dispatch_date:date,promised_delivery_date:date,actual...


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=3277]
   +- FileScan csv [shipment_id#17,booking_date#18,dispat

In [52]:
#26
shipment_and_warehouse_df1 = shipments_df.join(
    warehouse_df,
    on = 'warehouse_id',
    how = 'inner'
)
shipment_and_warehouse_df1.repartition(4).write.mode('overwrite').parquet('shipment_and_warehouse_df1.parquet')

**FINAL CHALLENGE | Logistics Performance Pipeline**

In [53]:
from pyspark.sql.functions import (
    col, when, sum, avg, count, datediff, to_date, dense_rank
)
from pyspark.sql.window import Window

# 1. Clean missing values.
# Strategy: use zero for missing cost and weight for this exercise.
clean_shipments = shipments_df.fillna(
    0, subset=['shipping_cost_inr', 'package_weight_kg']
).withColumn(
    'dispatch_date', to_date('dispatch_date')
).withColumn(
    'actual_delivery_date', to_date('actual_delivery_date')
).withColumn(
    'promised_delivery_date', to_date('promised_delivery_date')
)

# 2
prepared_df = clean_shipments.withColumn(
    'delivery_duration_days',
    datediff('actual_delivery_date', 'dispatch_date')
).withColumn(
    'delivery_performance',
    when(col('actual_delivery_date').isNull(), 'Missing actual date')
    .when(
        col('actual_delivery_date') < col('promised_delivery_date'),
        'Early'
    )
    .when(
        col('actual_delivery_date') == col('promised_delivery_date'),
        'On Time'
    )
    .otherwise('Late')
).withColumn(
    'is_late',
    when(col('delivery_performance') == 'Late', 1).otherwise(0)
).withColumn(
    'is_on_time',
    when(col('delivery_performance').isin('Early', 'On Time'), 1)
    .otherwise(0)
).withColumn(
    'shipment_profit',
    col('shipping_fee_inr') - col('shipping_cost_inr')
)

# 3
joined_df = prepared_df.join(
    warehouse_df, on='warehouse_id', how='inner'
).join(
    carrier_df, on='carrier_id', how='inner'
).join(
    routes_df, on='route_id', how='inner'
)

# 4
report_df = joined_df.groupBy(
    'region', 'carrier_id', 'carrier_name'
).agg(
    sum('shipping_fee_inr').alias('total_revenue'),
    sum('shipment_profit').alias('total_profit'),
    (avg('is_late') * 100).alias('late_delivery_pct'),
    (avg('is_on_time') * 100).alias('on_time_delivery_pct'),
    count('shipment_id').alias('shipment_count')
)

# 5
window_spec = Window.partitionBy('region').orderBy(
    col('on_time_delivery_pct').desc()
)

final_report_df = report_df.withColumn(
    'carrier_rank',
    dense_rank().over(window_spec)
)

# 6
final_report_df.cache()

final_report_df.count()
final_report_df.orderBy('region', 'carrier_rank').show()

# 7
final_report_df.write.mode('overwrite').partitionBy(
    'region'
).parquet('logistics_performance_report.parquet')

final_report_df.unpersist()

+------+----------+--------------------+------------------+------------------+------------------+--------------------+--------------+------------+
|region|carrier_id|        carrier_name|     total_revenue|      total_profit| late_delivery_pct|on_time_delivery_pct|shipment_count|carrier_rank|
+------+----------+--------------------+------------------+------------------+------------------+--------------------+--------------+------------+
|  East|       C07|   BluePeak Delivery| 91602.35999999997|          21196.56|27.490039840637447|   43.02788844621514|           251|           1|
|  East|       C01|   SwiftLine Express|         156652.36| 40569.96000000003|28.467153284671532|  41.605839416058394|           411|           2|
|  East|       C03|MetroRoute Logistics|129579.09999999989|36595.199999999975| 28.52941176470588|  40.588235294117645|           340|           3|
|  East|       C02|        ParcelBridge|157422.92000000007| 41168.72000000003|29.723502304147466|   40.09216589861751|

DataFrame[region: string, carrier_id: string, carrier_name: string, total_revenue: double, total_profit: double, late_delivery_pct: double, on_time_delivery_pct: double, shipment_count: bigint, carrier_rank: int]